In [16]:
import pandas as pd
import numpy as np
import itertools
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, HashingVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn import metrics
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("Flirt_dataset.txt",header=None,on_bad_lines="skip",encoding='utf8')

In [3]:
df=df.drop(0)
df.columns=['Date','Chat']
Message=df["Chat"].str.split("-",n=1,expand=True)
df["Time"]=Message[0]
Message1=Message[1].str.split(":",n=1,expand=True)
df["Name"]=Message1[0]
df["Chat"]=Message1[1]
df=df[["Date","Time","Name","Chat"]]
df

,Date,Time,Name,Chat
1,5/5/26,8:00 AM,karthick,Hi
2,5/5/26,8:00 AM,karthick,Good morning!
3,5/5/26,8:00 AM,karthick,what doing?
4,5/5/26,8:05 AM,sofi,Hi da
5,5/5/26,8:05 AM,sofi,good morning . getting ready to go out. how a...
...,...,...,...,...
98,6/29/26,7:55 AM,karthick,Definitely beautiful.
99,6/29/26,8:00 AM,sofi,Waiting in the lobby. come soon.
100,6/29/26,9:50 PM,karthick,Missed video call
101,6/29/26,10:10 PM,karthick,Missed video call


In [4]:
df["Label"] = 0

In [5]:
df.loc[[13, 21, 22, 26, 34, 40, 41, 44, 51, 52, 53, 55, 59, 68, 73, 75, 76, 77, 78, 80, 81, 83, 84, 85, 86, 88, 87, 93, 94, 98], "Label"] = 1

In [6]:
df

,Date,Time,Name,Chat,Label
1,5/5/26,8:00 AM,karthick,Hi,0
2,5/5/26,8:00 AM,karthick,Good morning!,0
3,5/5/26,8:00 AM,karthick,what doing?,0
4,5/5/26,8:05 AM,sofi,Hi da,0
5,5/5/26,8:05 AM,sofi,good morning . getting ready to go out. how a...,0
...,...,...,...,...,...
98,6/29/26,7:55 AM,karthick,Definitely beautiful.,1
99,6/29/26,8:00 AM,sofi,Waiting in the lobby. come soon.,0
100,6/29/26,9:50 PM,karthick,Missed video call,0
101,6/29/26,10:10 PM,karthick,Missed video call,0


In [7]:
df.to_csv("flirt_dataset_labeled.csv", index=False)

In [8]:
X = df["Chat"]
y = df["Label"]

In [9]:
X_train, X_test, y_train, y_test = train_test_split(df['Chat'], y, test_size=0.33, random_state=53)

# Trying Count Vectorizer

# 1. Count Vectorizer with Naive Bayes

In [10]:
count_vectorizer = CountVectorizer(stop_words='english')
count_train = count_vectorizer.fit_transform(X_train)
print(count_train)


<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 152 stored elements and shape (68, 76)>
  Coords	Values
  (0, 43)	1
  (1, 12)	1
  (1, 66)	1
  (2, 33)	1
  (2, 25)	1
  (2, 45)	1
  (3, 22)	1
  (3, 56)	1
  (4, 11)	1
  (4, 54)	1
  (4, 15)	1
  (5, 66)	1
  (5, 32)	1
  (5, 39)	1
  (5, 7)	1
  (6, 44)	1
  (6, 69)	1
  (7, 72)	1
  (7, 10)	1
  (8, 25)	1
  (8, 45)	1
  (9, 33)	1
  (9, 25)	1
  (9, 45)	1
  (9, 3)	1
  :	:
  (57, 23)	1
  (57, 35)	1
  (58, 72)	1
  (59, 33)	1
  (59, 25)	1
  (59, 45)	1
  (59, 3)	1
  (60, 42)	1
  (60, 55)	1
  (60, 52)	1
  (60, 0)	1
  (61, 60)	1
  (62, 9)	1
  (62, 24)	1
  (63, 33)	1
  (63, 25)	1
  (63, 45)	1
  (64, 41)	1
  (64, 48)	1
  (65, 75)	1
  (66, 44)	1
  (66, 69)	1
  (67, 66)	1
  (67, 37)	1
  (67, 26)	1


In [11]:
count_test = count_vectorizer.transform(X_test)
count_test

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 57 stored elements and shape (34, 76)>

In [12]:
len(count_vectorizer.get_feature_names_out())

76

In [13]:
clf = MultinomialNB()

clf.fit(count_train, y_train)
pred = clf.predict(count_test)
score = metrics.accuracy_score(y_test, pred)
print("accuracy:   %0.3f" % score)

cm = metrics.confusion_matrix(y_test, pred, labels=['0', '1'])

accuracy:   0.794


In [14]:
report=classification_report(y_test, pred)
print(report)

              precision    recall  f1-score   support

           0       0.81      0.96      0.88        27
           1       0.50      0.14      0.22         7

    accuracy                           0.79        34
   macro avg       0.66      0.55      0.55        34
weighted avg       0.75      0.79      0.75        34



# Accuracy - 0.794

# 2. Count Vectorizer with Logistic Regression

In [17]:
lr = LogisticRegression(max_iter=1000)

lr.fit(count_train, y_train)

pred = lr.predict(count_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Accuracy: 0.7941176470588235
              precision    recall  f1-score   support

           0       0.81      0.96      0.88        27
           1       0.50      0.14      0.22         7

    accuracy                           0.79        34
   macro avg       0.66      0.55      0.55        34
weighted avg       0.75      0.79      0.75        34



# Accuracy - 0.7941

# 3. Count Vectorizer with Linear SVM

In [18]:
svm = LinearSVC()

svm.fit(count_train, y_train)

pred = svm.predict(count_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Accuracy: 0.7058823529411765
              precision    recall  f1-score   support

           0       0.79      0.85      0.82        27
           1       0.20      0.14      0.17         7

    accuracy                           0.71        34
   macro avg       0.50      0.50      0.49        34
weighted avg       0.67      0.71      0.69        34



# Accuracy - 0.7058

# Trying TF-IDF Vectorizer

# 4. TF-IDF with Naive Bayes

In [19]:
tfidf = TfidfVectorizer()

tfidf_train = tfidf.fit_transform(X_train)

tfidf_test = tfidf.transform(X_test)

In [20]:
nb = MultinomialNB()

nb.fit(tfidf_train, y_train)

pred = nb.predict(tfidf_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Accuracy: 0.8529411764705882
              precision    recall  f1-score   support

           0       0.84      1.00      0.92        27
           1       1.00      0.29      0.44         7

    accuracy                           0.85        34
   macro avg       0.92      0.64      0.68        34
weighted avg       0.88      0.85      0.82        34



# Accuracy - 0.8529

# 5. TF-IDF with Logistic Regression

In [21]:
lr = LogisticRegression(max_iter=1000)

lr.fit(tfidf_train, y_train)

pred = lr.predict(tfidf_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Accuracy: 0.8823529411764706
              precision    recall  f1-score   support

           0       0.87      1.00      0.93        27
           1       1.00      0.43      0.60         7

    accuracy                           0.88        34
   macro avg       0.94      0.71      0.77        34
weighted avg       0.90      0.88      0.86        34



# Accuracy - 0.8823

# 6. TF-IDF with Linear SVM

In [22]:
svm = LinearSVC()

svm.fit(tfidf_train, y_train)

pred = svm.predict(tfidf_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Accuracy: 0.8823529411764706
              precision    recall  f1-score   support

           0       0.93      0.93      0.93        27
           1       0.71      0.71      0.71         7

    accuracy                           0.88        34
   macro avg       0.82      0.82      0.82        34
weighted avg       0.88      0.88      0.88        34



# Accuracy - 0.8823

TF-IDF with Linear SVM and TF-IDF with Logistic Regression achieved the highest accuracy of 88.23%. 

Recall (0.71) and F1-score (0.71) for the Flirt class (1) is higher for Linear SVM  

This indicates that the SVM model is more effective at identifying flirt messages while maintaining the same overall accuracy.

Final model - TF-IDF with Linear SVM

# Whatsapp chat Analysis

In [23]:
chat_tfidf = tfidf.transform(df["Chat"])

df["Prediction"] = svm.predict(chat_tfidf)

In [24]:
#df["Prediction"] = df["Prediction"].map({0: "Non-Flirt",1: "Flirt"})

In [25]:
df

,Date,Time,Name,Chat,Label,Prediction
1,5/5/26,8:00 AM,karthick,Hi,0,0
2,5/5/26,8:00 AM,karthick,Good morning!,0,0
3,5/5/26,8:00 AM,karthick,what doing?,0,0
4,5/5/26,8:05 AM,sofi,Hi da,0,0
5,5/5/26,8:05 AM,sofi,good morning . getting ready to go out. how a...,0,1
...,...,...,...,...,...,...
98,6/29/26,7:55 AM,karthick,Definitely beautiful.,1,1
99,6/29/26,8:00 AM,sofi,Waiting in the lobby. come soon.,0,0
100,6/29/26,9:50 PM,karthick,Missed video call,0,0
101,6/29/26,10:10 PM,karthick,Missed video call,0,0


In [26]:
df["Name"].value_counts()

 karthick    67
 sofi        35
Name: Name, dtype: int64

# Flirt Encounters

Most Talkative Person

In [27]:
df["Name"].value_counts().idxmax()

' karthick'

Less Talkative Person

In [28]:
df["Name"].value_counts().idxmin()

' sofi'

# Time Encounters

Most Active Date

In [29]:
most_active_date = df["Date"].value_counts().idxmax()

print("Most Active Date:", most_active_date)

Most Active Date: 5/14/26


Most Active Day

In [30]:
date_obj = pd.to_datetime(most_active_date, format="%m/%d/%y")

print("Most Active Day:", date_obj.day_name())

Most Active Day: Thursday


Most Active Time

In [31]:
active_date_df = df[df["Date"] == most_active_date].copy()

In [32]:
active_date_df["Time"] = (active_date_df["Time"].str.replace("\u202f", " ", regex=False) .str.strip())

In [33]:
active_date_df["Hour"] = active_date_df["Time"].str.extract(r'(\d+):').astype(int)

active_date_df["Period"] = active_date_df["Time"].str.extract(r'(AM|PM)')

In [34]:
active_date_df.loc[(active_date_df["Period"] == "PM") & (active_date_df["Hour"] != 12),"Hour"] += 12

active_date_df.loc[(active_date_df["Period"] == "AM") & (active_date_df["Hour"] == 12),"Hour"] = 0

In [35]:
most_active_hour = active_date_df["Hour"].value_counts().idxmax()

In [36]:
if most_active_hour == 0:
    display_time = "12 AM"
elif most_active_hour < 12:
    display_time = f"{most_active_hour} AM"
elif most_active_hour == 12:
    display_time = "12 PM"
else:
    display_time = f"{most_active_hour-12} PM"

print("Most Active Hour :", display_time)

Most Active Hour : 8 AM


Avg no of msg per day

In [37]:
avg_messages = df.groupby("Date").size().mean()

print("Average messages per day:", avg_messages)

Average messages per day: 6.8


# Media Encounters

Media Count

In [38]:
media_count = df["Chat"].str.contains("<Media omitted>", case=False, na=False).sum()

print("Media Count:", media_count)

Media Count: 10


In [39]:
media_person = df[df["Chat"].str.contains("<Media omitted>", case=False, na=False)]

print(media_person["Name"].value_counts())

 karthick    9
 sofi        1
Name: Name, dtype: int64


Deleted Message Count

In [40]:
deleted_count = df["Chat"].str.contains("deleted", case=False, na=False).sum()

print("Deleted Message Count:", deleted_count)

Deleted Message Count: 0


Missed Voice Call Count

In [41]:
missed_voice = df["Chat"].str.contains("Missed voice call", case=False, na=False).sum()

print("Missed Voice Call Count:", missed_voice)

Missed Voice Call Count: 2


Missed Video Call Count

In [42]:
missed_video = df["Chat"].str.contains("Missed video call", case=False, na=False).sum()

print("Missed Video Call Count:", missed_video)

Missed Video Call Count: 6
